# Conclusiones de mineria de datos

In [172]:
import pandas as pd


#### - ¿Donde se aconseja realizar un explotacion agrícola para obtener un beneficio entre 5 % y 10 %?

Para el calculo de los beneficios se necesitan los ingresos (producción por precio de venta tonelada) y los costes, para los costes necesitamos el coste del fertilizante, vamos a colocar una estimación de 1.20 USD por kg, el coste del agua que vamoas a estimar una coste de agua de 0.05 usd por m3 y vamos a asumir unos costes fijos base (mano de obra, maquinaria, semillas) de unos 300 usd.

In [173]:
df_precio = pd.read_csv('datos/precios_mercado_limpios.csv')
df_agricola = pd.read_csv('datos/produccion_agricola.csv')

In [174]:
df_agricola.head()


,pais,codigo_iso,region,cultivo,anio,superficie_hectareas,rendimiento_ton_ha,produccion_ton,fertilizantes_kg_ha,agua_riego_m3_ha,tendencia_5_anios
0,Argentina,ARG,América del Sur,Soja,2011,4703643,2.22,10465461,105,4811,0.000536
1,Argentina,ARG,América del Sur,Maíz,2011,3371781,8.45,28485921,164,5557,0.030943
2,Argentina,ARG,América del Sur,Cebada,2011,1661577,2.77,4596917,105,6514,-0.039779
3,Argentina,ARG,América del Sur,Té,2011,188518,3.07,577886,236,8527,0.003623
4,Argentina,ARG,América del Sur,Algodón,2011,1638327,1.18,1929756,276,6925,0.007735


Preparacion de los precios

In [175]:
df_precios_medios = df_precio.groupby(['pais', 'producto'])['precio_usd_ton'].mean().reset_index()
# Renombramos la columna para que coincida con el dataset agrícola al hacer el cruce
df_precios_medios.rename(columns={'producto': 'cultivo'}, inplace=True)

Preparación de la produccion (media historica por pais y cultivo)

In [176]:
df_ag_media = df_agricola.groupby(['pais', 'cultivo']).agg({
    'superficie_hectareas': 'mean',
    'produccion_ton': 'mean',
    'fertilizantes_kg_ha': 'mean',
    'agua_riego_m3_ha': 'mean'
}).reset_index()

Cruce de datos

In [177]:
df_rentabilidad = pd.merge(df_ag_media, df_precios_medios, on=['pais', 'cultivo'], how='inner')

In [178]:
df_rentabilidad.head()

,pais,cultivo,superficie_hectareas,produccion_ton,fertilizantes_kg_ha,agua_riego_m3_ha,precio_usd_ton
0,Alemania,Maíz,5356857.1,40958173.4,206.3,5984.1,202.710000
1,Alemania,Soja,4763385.6,18714150.4,145.9,4784.5,513.152308
2,Alemania,Trigo,2464869.5,13072008.7,168.5,4391.1,314.085000
3,Argentina,Maíz,5215900.8,30937311.7,185.4,5699.9,201.473333
4,Argentina,Soja,3746825.4,11778393.7,184.6,4787.8,502.715714


Variables de costes

In [179]:
PRECIO_FERTILIZANTE_KG = 1.2
PRECIO_AGUA_M3 = 0.01
COSTE_BASE_HA = 650
COSTE_LOGISTICO_TON = 90.0


Calculo de ingresos y costes

In [180]:
# Ingresos = Producción (ton) * Precio (USD/ton)
df_rentabilidad['ingresos_usd'] = df_rentabilidad['produccion_ton'] * df_rentabilidad['precio_usd_ton']

# Costes de Estructura e Insumos
coste_insumos_ha = (COSTE_BASE_HA + 
                   (df_rentabilidad['fertilizantes_kg_ha'] * PRECIO_FERTILIZANTE_KG) + 
                   (df_rentabilidad['agua_riego_m3_ha'] * PRECIO_AGUA_M3))

# Coste total = (Coste/ha * superficie) + (Coste/ton * producción)
df_rentabilidad['costes_usd'] = (coste_insumos_ha * df_rentabilidad['superficie_hectareas']) + \
                               (df_rentabilidad['produccion_ton'] * COSTE_LOGISTICO_TON)

In [181]:
df_rentabilidad.head()

,pais,cultivo,superficie_hectareas,produccion_ton,fertilizantes_kg_ha,agua_riego_m3_ha,precio_usd_ton,ingresos_usd,costes_usd
0,Alemania,Maíz,5356857.1,40958173.4,206.3,5984.1,202.710000,8.302631e+09,8.814896e+09
1,Alemania,Soja,4763385.6,18714150.4,145.9,4784.5,513.152308,9.603209e+09,5.842352e+09
2,Alemania,Trigo,2464869.5,13072008.7,168.5,4391.1,314.085000,4.105722e+09,3.385277e+09
3,Argentina,Maíz,5215900.8,30937311.7,185.4,5699.9,201.473333,6.233043e+09,7.632428e+09
4,Argentina,Soja,3746825.4,11778393.7,184.6,4787.8,502.715714,5.921184e+09,4.504879e+09


Beneficio y margen

In [182]:
df_rentabilidad['beneficio_neto_usd'] = df_rentabilidad['ingresos_usd'] - df_rentabilidad['costes_usd']
df_rentabilidad['margen_beneficio_pct'] = (df_rentabilidad['beneficio_neto_usd'] / df_rentabilidad['ingresos_usd']) * 100

In [183]:
df_rentabilidad.describe()

,superficie_hectareas,produccion_ton,fertilizantes_kg_ha,agua_riego_m3_ha,precio_usd_ton,ingresos_usd,costes_usd,beneficio_neto_usd,margen_beneficio_pct
count,3.500000e+01,3.500000e+01,35.000000,35.000000,35.000000,3.500000e+01,3.500000e+01,3.500000e+01,35.000000
mean,3.290155e+06,1.708994e+07,175.525714,5548.894286,340.031143,5.282575e+09,4.559382e+09,7.231926e+08,14.090357
std,1.392442e+06,9.194822e+06,29.495945,777.369829,107.414279,2.036049e+09,2.033897e+09,1.195455e+09,18.385396
min,9.996325e+05,5.378905e+06,119.400000,4003.100000,190.684375,1.884100e+09,1.390377e+09,-1.462227e+09,-22.451071
25%,1.871148e+06,1.084341e+07,149.550000,4786.150000,288.437273,3.671630e+09,2.713718e+09,-7.411797e+07,-1.158815
50%,3.238336e+06,1.401752e+07,179.800000,5809.700000,312.213000,4.851376e+09,4.498039e+09,7.204444e+08,17.547326
75%,4.474818e+06,1.859570e+07,198.450000,6034.400000,372.070667,6.335279e+09,6.137416e+09,1.171370e+09,28.259366
max,5.901857e+06,4.122392e+07,240.200000,6904.400000,537.333333,9.603209e+09,9.322985e+09,3.760858e+09,39.162507


Filtrar entre 5% y 10%

In [184]:
# 1. Agrupar la rentabilidad haciendo la media por país
df_pais_rentabilidad = df_rentabilidad.groupby('pais').agg({
    'margen_beneficio_pct': 'mean',
    'ingresos_usd': 'sum',
    'costes_usd': 'sum'
}).reset_index()

# 2. Filtrar el objetivo macroeconómico (5% al 10%)
paises_ideales = df_pais_rentabilidad[
    (df_pais_rentabilidad['margen_beneficio_pct'] >= 5) & 
    (df_pais_rentabilidad['margen_beneficio_pct'] <= 10)
].sort_values(by='margen_beneficio_pct', ascending=False)

print("--- UBICACIONES MACROECONÓMICAS RECOMENDADAS (5% - 10%) ---")
print(paises_ideales[['pais', 'margen_beneficio_pct']])

--- UBICACIONES MACROECONÓMICAS RECOMENDADAS (5% - 10%) ---
             pais  margen_beneficio_pct
2       Australia              9.548509
11  Nueva Zelanda              5.804472
1       Argentina              5.335645


Los paises donde se recomienda hacer una explotación agricola asumiendo que se van a plantar varios tipos de productos, y donde se busca que tengan un beneficion total de entre el 5% y 10% son: australia, nueva zelanda y argentina. Para realizar esta afirmación tuvimos que calcular los ingresos (produccion (ton) * precio venta) y los costes (busqueda para asumir ciertos valores: costes fijos de superficie 650 USD, alquiler, seguros, maquinaria y preparacion del suelo; costes variables: 1.20 USD por kg de fertilizante y 0.01 USD por m3 de agua; y costes logisticos y de comercializacion: 90 USD por tonelada producida).

- Opción Primaria: Australia (Margen proyectado: 9.54%)
Se posiciona como la recomendación principal, rozando el límite superior del objetivo. Como ya demostró el modelo de Clustering (donde Australia formaba su propio clúster de "Gigante Extensivo"), su altísima eficiencia agrícola le permite absorber la estructura de costes fijos e insumos con gran facilidad, maximizando el valor de sus exportaciones en el mercado internacional.
- Opciones Secundarias: Nueva Zelanda (5.80%) y Argentina (5.33%)
Ambos países se sitúan en el límite inferior del rango objetivo. Representan opciones viables y seguras, caracterizadas por pertenecer a clústeres de gran especialización agro-ganadera y extensiva en el hemisferio sur. Su consolidación logística y acceso directo a puertos de exportación permiten sostener márgenes positivos constantes a pesar de los altos costes de transporte (90 USD/ton).
